# Assignment 15 — Smart Health Assistant (Sequential Conversation Pattern, AG2)

**Objective:** Build a Smart Health Assistant using a **sequential conversation pattern**. A **User Proxy Agent** collects the user's data and shares it with three LLM agents in a fixed order: **BMI Agent** (uses a registered **BMI tool**) → **Diet Planner Agent** → **Workout Scheduler Agent**. The final output is a complete health plan: BMI insights, a tailored diet, and a weekly fitness schedule.

**Framework:** AG2 `1.1.2` (`pip install ag2[openai]`) — the current AG2 release. Since AG2 v1.0 (July 2026) the top-level package is the protocol-driven API (`from ag2 import Agent, tool`); the older `autogen.*` classes (`ConversableAgent`, `UserProxyAgent`, `initiate_chats`) now live in the separate `ag2-classic` package. This notebook uses the current API and maps the assignment's concepts onto it:

| Assignment concept | AG2 Classic (course handouts) | This notebook (AG2 1.x) |
|---|---|---|
| User Proxy Agent (collects data, no LLM) | `UserProxyAgent(llm_config=False)` | `UserProxyAgent` class: collects + validates inputs, sends each message, logs the transcript |
| LLM agent with a role | `AssistantAgent(system_message=...)` | `Agent(name, prompt=..., config=OpenAIConfig(...))` |
| Register a tool to an agent | `@bmi_agent.register_for_llm` / `register_for_execution` | `tool(calculate_bmi)` passed as `tools=[...]` to the BMI Agent only |
| Sequential chat | `autogen.initiate_chats([...])` | The User Proxy calls `initiate_chat` on each agent in order, passing the previous reply forward |

### How to run (Google Colab)
1. **Colab Secrets (recommended):** click the 🔑 icon in the left sidebar, add a secret named `OPENAI_API_KEY`, and turn on *Notebook access*.
2. **No secret?** Just run the notebook. If no key is found you will be prompted to paste one (hidden input, never printed or saved). An `OPENAI_API_KEY` environment variable also works.
3. `Runtime → Run all`. In the data-collection cell, press **Enter** to accept each default value or type your own. The last cell prints a PASS/FAIL rubric.

### Rubric (verified automatically in the last cell)

| # | Expectation (from the assignment) | Where it is satisfied |
|---|---|---|
| R1 | Agents run in the stated **sequence**: BMI Agent → Diet Planner → Workout Scheduler | Sequential chat cell |
| R2 | **User Proxy Agent collects** weight (kg), height (cm), age, gender, dietary preference | User Proxy cell |
| R3 | User Proxy **validates** input and rejects bad values (should-fail case) | User Proxy cell self-test |
| R4 | **BMI tool** converts cm → m and computes `kg / m²` correctly (known values + should-fail cases) | BMI tool cell self-test |
| R5 | BMI tool is **registered to the BMI Agent** (and only to it) | Agent definitions cell |
| R6 | BMI Agent **actually calls the tool** and reports the tool's BMI value | Transcript + rubric |
| R7 | BMI Agent gives the **BMI category** and **health recommendations** | BMI Agent prompt + output |
| R8 | Diet Planner receives **BMI insights + dietary preference** | Sequential chat cell |
| R9 | Diet Planner produces a **tailored meal plan** | Diet Planner output |
| R10 | Workout Scheduler receives **meal plan + age + gender** | Sequential chat cell |
| R11 | Workout Scheduler produces a **weekly workout plan** | Workout Scheduler output |
| R12 | Final output is a **complete health plan** (BMI insights + diet + fitness schedule) in Markdown | Final output cell |


## 1. Install dependencies
Installed with **uv** (fast). Versions are pinned so the notebook stays reproducible.

In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python "{sys.executable}" "ag2[openai]==1.1.2"

import ag2
print("AG2 version:", ag2.__version__)

## 2. API key
Looks for the key in **Colab Secrets** first, then an environment variable, then asks you to paste it. The key is never printed.

In [ ]:
import os
from getpass import getpass


def load_api_key() -> str:
    try:
        from google.colab import userdata  # only exists inside Colab
        key = userdata.get("OPENAI_API_KEY")
        if key:
            return key
    except Exception:
        pass  # not in Colab, secret missing, or notebook access not granted
    key = os.environ.get("OPENAI_API_KEY")
    if key:
        return key
    return getpass("OPENAI_API_KEY not found in Colab Secrets or environment. Paste your key: ")


API_KEY = load_api_key().strip()
if not API_KEY:
    raise ValueError("No API key provided. Add OPENAI_API_KEY to Colab Secrets or paste it when prompted.")
print("API key loaded.")

## 3. Settings and model
All settings live here. The first reachable model in `MODEL_CANDIDATES` is used, so the notebook works even if your account lacks the newest model. Set `INTERACTIVE_INPUT = False` to skip the prompts and use `DEFAULT_PROFILE` (useful for unattended runs).

In [ ]:
from ag2 import Agent
from ag2.config import OpenAIConfig

MODEL_CANDIDATES = ["gpt-5.4-mini", "gpt-5-mini", "gpt-4.1-mini"]  # tried in order; first reachable wins

INTERACTIVE_INPUT = True  # True: prompt for each value (Enter accepts the default). False: use DEFAULT_PROFILE as-is.
DEFAULT_PROFILE = {"weight_kg": 82, "height_cm": 175, "age": 34, "gender": "Male", "diet_preference": "Veg"}


async def pick_model() -> str:
    """Fail fast with a clear message if the key or every model name is wrong."""
    for model in MODEL_CANDIDATES:
        try:
            probe = Agent("probe", prompt="Reply with the single word: OK", config=OpenAIConfig(model=model, api_key=API_KEY))
            reply = await probe.ask("Ping")
            print(f"Model '{model}' reachable (test reply: {str(reply.body).strip()[:10]!r})")
            return model
        except Exception as exc:  # the provider error text can echo part of the key, so print only the error type
            print(f"Model '{model}' not usable ({type(exc).__name__}); trying the next candidate...")
    raise RuntimeError("No candidate model is reachable. Check that the API key is valid and MODEL_CANDIDATES "
                       "contains a model your account can use.")


MODEL = await pick_model()


def make_config() -> OpenAIConfig:
    """One config per agent. Change MODEL above to switch every agent at once."""
    return OpenAIConfig(model=MODEL, api_key=API_KEY)

## 4. Step 1 — User Proxy Agent
Collects all inputs from the user, **validates** them with explicit rules (type, range, allowed values — no LLM involved), and acts as the *sender* of every message in the sequential chat, just like `UserProxyAgent` in AG2 Classic. `initiate_chat` prints the exchange (including any tool calls the agent made) and keeps a log that the rubric checks later.

In [ ]:
from dataclasses import dataclass, asdict

from ag2.events import ToolCallEvent, ToolResultEvent

VALID_GENDERS = ("Male", "Female", "Other")
VALID_DIETS = ("Veg", "Non-Veg", "Vegan")


@dataclass(frozen=True)
class UserProfile:
    weight_kg: float
    height_cm: float
    age: int
    gender: str
    diet_preference: str


def validate_profile(weight_kg, height_cm, age, gender, diet_preference) -> UserProfile:
    """Deterministic validation: reject anything outside sane ranges or allowed values."""
    weight_kg, height_cm, age = float(weight_kg), float(height_cm), int(age)
    if not 20 <= weight_kg <= 400:
        raise ValueError(f"weight_kg must be between 20 and 400, got {weight_kg}")
    if not 100 <= height_cm <= 250:
        raise ValueError(f"height_cm must be between 100 and 250, got {height_cm}")
    if not 10 <= age <= 120:
        raise ValueError(f"age must be between 10 and 120, got {age}")
    gender = gender.strip().capitalize()
    if gender not in VALID_GENDERS:
        raise ValueError(f"gender must be one of {VALID_GENDERS}, got {gender!r}")
    diet_map = {d.lower(): d for d in VALID_DIETS}
    diet = diet_preference.strip().lower().replace(" ", "-")
    if diet not in diet_map:
        raise ValueError(f"diet_preference must be one of {VALID_DIETS}, got {diet_preference!r}")
    return UserProfile(weight_kg, height_cm, age, gender, diet_map[diet])


class UserProxyAgent:
    """Collects user data (no LLM) and drives the sequential conversation."""

    name = "User_Proxy_Agent"

    def __init__(self, interactive: bool, defaults: dict):
        self.interactive = interactive
        self.defaults = defaults
        self.chat_log: list[dict] = []  # one entry per hop; inspected by the rubric

    def _ask(self, label: str, default) -> str:
        raw = input(f"{label} [{default}]: ").strip() if self.interactive else ""
        return raw or str(default)

    def collect(self) -> UserProfile:
        """Step 1: collect the five inputs, re-prompting up to 3 times on invalid values."""
        print(f"[{self.name}] Collecting user data (press Enter to accept a default)")
        for attempt in range(3):
            try:
                profile = validate_profile(
                    self._ask("(i)   Weight in kg", self.defaults["weight_kg"]),
                    self._ask("(ii)  Height in cm", self.defaults["height_cm"]),
                    self._ask("(iii) Age", self.defaults["age"]),
                    self._ask(f"(iv)  Gender {VALID_GENDERS}", self.defaults["gender"]),
                    self._ask(f"(v)   Dietary preference {VALID_DIETS}", self.defaults["diet_preference"]),
                )
                print(f"[{self.name}] Collected: {asdict(profile)}")
                return profile
            except ValueError as exc:
                if not self.interactive or attempt == 2:
                    raise
                print(f"  Invalid input: {exc}. Please try again.\n")

    async def initiate_chat(self, recipient: Agent, message: str):
        """Send one message to an agent, print the exchange like AG2 Classic, and log it."""
        line = "-" * 80
        print("\n" + "*" * 80 + f"\n{self.name} -> {recipient.name}\n{line}\n{message}\n{line}")
        reply = await recipient.ask(message)

        tool_calls, pending = [], {}
        for event in await reply.history.get_events():
            if isinstance(event, ToolCallEvent):
                pending[event.id] = event
                print(f">>> {recipient.name} called tool `{event.name}` with arguments {event.serialized_arguments}")
            elif isinstance(event, ToolResultEvent) and event.parent_id in pending:
                call = pending.pop(event.parent_id)
                result = [getattr(p, "data", getattr(p, "content", p)) for p in event.result.parts]
                result = result[0] if len(result) == 1 else result
                print(f">>> tool `{event.name}` returned {result}")
                tool_calls.append({"name": call.name, "arguments": call.serialized_arguments, "result": result})

        print(f"{line}\n{recipient.name} -> {self.name}\n{line}\n{reply.body}\n")
        self.chat_log.append({"sender": self.name, "recipient": recipient.name, "message": message,
                              "reply": reply.body or "", "tool_calls": tool_calls})
        return reply


# Self-test: valid input passes, bad input must fail (should-fail cases).
assert validate_profile(70, 175, 30, "female", "vegan") == UserProfile(70.0, 175.0, 30, "Female", "Vegan")
VALIDATION_SHOULD_FAIL_OK = True
for bad in [(0, 175, 30, "Male", "Veg"), (70, 175, 300, "Male", "Veg"), (70, 175, 30, "Robot", "Veg"), (70, 175, 30, "Male", "Keto")]:
    try:
        validate_profile(*bad)
        VALIDATION_SHOULD_FAIL_OK = False
        print("FAIL: accepted bad input", bad)
    except ValueError:
        pass
print("User Proxy Agent defined; validation self-test:", "PASS" if VALIDATION_SHOULD_FAIL_OK else "FAIL")

## 5. Step 2 — BMI tool
Plain Python, no LLM: `height_m = height_cm / 100`, then `BMI = weight_kg / height_m²`. The tool is unit-tested here (known values + should-fail cases) and wrapped with AG2's `tool()` so the BMI Agent can call it. (`@tool` as a decorator is the equivalent shorthand.)

In [ ]:
from ag2 import tool


def calculate_bmi(weight_kg: float, height_cm: float) -> dict:
    """Calculate the Body Mass Index (BMI) from weight in kilograms and height in centimetres.

    Converts height from cm to m (height_m = height_cm / 100) and returns BMI = weight_kg / height_m ** 2,
    rounded to one decimal, together with the inputs used.
    """
    if weight_kg <= 0 or height_cm <= 0:
        raise ValueError("weight_kg and height_cm must be positive numbers")
    height_m = height_cm / 100
    bmi = round(weight_kg / (height_m ** 2), 1)
    return {"weight_kg": weight_kg, "height_cm": height_cm, "height_m": height_m, "bmi": bmi}


# Self-test: known values must match, invalid values must raise.
assert calculate_bmi(70, 175)["bmi"] == 22.9
assert calculate_bmi(82, 175)["bmi"] == 26.8
assert calculate_bmi(50, 180)["height_m"] == 1.8
TOOL_SELFTEST_OK = True
for bad in [(0, 175), (70, 0), (-70, 175), (70, -175)]:
    try:
        calculate_bmi(*bad)
        TOOL_SELFTEST_OK = False
        print("FAIL: tool accepted bad input", bad)
    except ValueError:
        pass

bmi_tool = tool(calculate_bmi)  # AG2 reads the name, docstring and type hints to build the tool schema for the LLM
print("BMI tool ready:", bmi_tool.name, "| self-test:", "PASS" if TOOL_SELFTEST_OK else "FAIL")

## 6. Steps 3–5 — BMI Agent, Diet Planner Agent, Workout Scheduler Agent
Each agent gets a clear role through its `prompt` (the system message). The BMI tool is registered **only** to the BMI Agent via `tools=[bmi_tool]`; the other agents have no tools.

In [ ]:
BMI_AGENT = "BMI_Agent"
DIET_AGENT = "Diet_Planner_Agent"
WORKOUT_AGENT = "Workout_Scheduler_Agent"

bmi_agent = Agent(
    BMI_AGENT,
    prompt="""You are the BMI Agent in a Smart Health Assistant. You receive the user's weight (kg) and height (cm).
Rules:
1. ALWAYS call the `calculate_bmi` tool to compute the BMI. Never compute or estimate it yourself.
2. Classify the tool's BMI with the WHO adult categories: Underweight < 18.5; Normal weight 18.5-24.9; Overweight 25.0-29.9; Obese >= 30.0.
3. Reply in Markdown with exactly these three lines/sections:
**BMI:** <the tool's value, one decimal>
**Category:** <category>
**Health recommendations:** 3-5 short bullet points tailored to that category (general wellness guidance, not a medical diagnosis).
Keep it under 200 words.""",
    config=make_config(),
    tools=[bmi_tool],
)

diet_planner_agent = Agent(
    DIET_AGENT,
    prompt="""You are the Diet Planner Agent. You receive BMI insights from the BMI Agent and the user's dietary preference
(Veg, Non-Veg or Vegan) from the User Proxy Agent.
Create a one-day meal plan (Breakfast, Lunch, Snack, Dinner) whose goal matches the BMI category
(calorie deficit for Overweight/Obese, maintenance for Normal weight, calorie surplus for Underweight) and that STRICTLY
respects the dietary preference: Veg = no meat, fish or eggs; Vegan = no animal products at all; Non-Veg = no restriction.
Reply in Markdown: first a line '**Goal:** ...' that names the BMI category and the dietary preference, then a table with
columns Meal | Foods | Why it fits. Keep it under 250 words.""",
    config=make_config(),
)

workout_scheduler_agent = Agent(
    WORKOUT_AGENT,
    prompt="""You are the Workout Scheduler Agent. You receive the meal plan from the Diet Planner Agent and the user's age and
gender from the User Proxy Agent.
Create a 7-day weekly workout schedule (Monday to Sunday) that supports the meal plan's goal and is appropriate for the
user's age and gender. Include at least one rest or active-recovery day.
Reply in Markdown: first a line '**Plan for:** age and gender', then a table with columns Day | Workout | Duration | Notes,
then 2-3 safety tips. Keep it under 250 words.""",
    config=make_config(),
)

for agent in (bmi_agent, diet_planner_agent, workout_scheduler_agent):
    print(f"{agent.name:<26} tools: {[t.name for t in agent.tools] or 'none'}")

## 7. Run the sequential conversation
The User Proxy Agent collects the data (Step 1) and then talks to each agent **in order**, carrying the previous agent's output forward — the sequential conversation pattern:

1. User Proxy → **BMI Agent** (weight, height) → calls the BMI tool → BMI insights
2. User Proxy → **Diet Planner Agent** (BMI insights + dietary preference) → meal plan
3. User Proxy → **Workout Scheduler Agent** (meal plan + age + gender) → weekly workout plan

In [ ]:
user_proxy = UserProxyAgent(interactive=INTERACTIVE_INPUT, defaults=DEFAULT_PROFILE)
profile = user_proxy.collect()  # Step 1

# Step 2 + 3: BMI tool (called by the agent) and BMI Agent
bmi_reply = await user_proxy.initiate_chat(
    bmi_agent,
    f"User data collected by the User Proxy Agent:\n- Weight: {profile.weight_kg} kg\n- Height: {profile.height_cm} cm\n"
    "Calculate my BMI with the calculate_bmi tool, classify it, and give health recommendations.",
)
bmi_insights = bmi_reply.body

# Step 4: Diet Planner Agent gets the BMI insights + dietary preference
diet_reply = await user_proxy.initiate_chat(
    diet_planner_agent,
    f"BMI insights from the BMI Agent:\n{bmi_insights}\n\nDietary preference (from the User Proxy Agent): {profile.diet_preference}\n"
    "Create a tailored meal plan.",
)
meal_plan = diet_reply.body

# Step 5: Workout Scheduler Agent gets the meal plan + age + gender
workout_reply = await user_proxy.initiate_chat(
    workout_scheduler_agent,
    f"Meal plan from the Diet Planner Agent:\n{meal_plan}\n\nUser details (from the User Proxy Agent):\n- Age: {profile.age}\n"
    f"- Gender: {profile.gender}\nCreate a weekly workout plan.",
)
workout_plan = workout_reply.body
print("Sequential conversation complete:", " -> ".join(entry["recipient"] for entry in user_proxy.chat_log))

## 8. Final output — the complete health plan (Markdown)

In [ ]:
from IPython.display import Markdown, display

final_plan = f"""# Smart Health Assistant — Personalized Health Plan

**Profile:** {profile.weight_kg} kg · {profile.height_cm} cm · age {profile.age} · {profile.gender} · diet: {profile.diet_preference}

---
## 1. BMI Insights (BMI Agent)
{bmi_insights}

---
## 2. Tailored Diet (Diet Planner Agent)
{meal_plan}

---
## 3. Weekly Fitness Schedule (Workout Scheduler Agent)
{workout_plan}
"""
display(Markdown(final_plan))

## 9. Rubric verification
Automated check of the rubric table at the top. Every row should print `PASS`.

In [ ]:
log = user_proxy.chat_log
bmi_entry, diet_entry, workout_entry = (log + [{}] * 3)[:3]
expected_bmi = round(profile.weight_kg / (profile.height_cm / 100) ** 2, 1)  # independent re-computation
tool_results = [c["result"] for c in bmi_entry.get("tool_calls", []) if c["name"] == "calculate_bmi"]
tool_bmi = tool_results[0].get("bmi") if tool_results and isinstance(tool_results[0], dict) else None
categories = ("underweight", "normal", "overweight", "obese")
days = ("monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday")
text = lambda s: (s or "").lower()

checks = [
    ("R1  Sequence is BMI Agent -> Diet Planner -> Workout Scheduler",
     [e.get("recipient") for e in log] == [BMI_AGENT, DIET_AGENT, WORKOUT_AGENT]),
    ("R2  User Proxy collected weight, height, age, gender, dietary preference",
     isinstance(profile, UserProfile) and profile.gender in VALID_GENDERS and profile.diet_preference in VALID_DIETS),
    ("R3  User Proxy validation rejects bad input (should-fail cases)", VALIDATION_SHOULD_FAIL_OK),
    ("R4  BMI tool formula correct (cm -> m, kg / m^2) incl. should-fail cases", TOOL_SELFTEST_OK),
    ("R5  BMI tool registered to the BMI Agent only",
     [t.name for t in bmi_agent.tools] == [bmi_tool.name] and not diet_planner_agent.tools and not workout_scheduler_agent.tools),
    ("R6  BMI Agent called the tool and reported its value",
     tool_bmi == expected_bmi and str(expected_bmi) in text(bmi_insights)),
    ("R7  BMI Agent gave a category and health recommendations",
     any(c in text(bmi_insights) for c in categories) and "recommend" in text(bmi_insights)),
    ("R8  Diet Planner received BMI insights + dietary preference",
     bmi_insights in diet_entry.get("message", "") and profile.diet_preference in diet_entry.get("message", "")),
    ("R9  Diet Planner produced a meal plan",
     sum(m in text(meal_plan) for m in ("breakfast", "lunch", "dinner")) >= 2 and profile.diet_preference.lower() in text(meal_plan)),
    ("R10 Workout Scheduler received meal plan + age + gender",
     meal_plan in workout_entry.get("message", "") and f"Age: {profile.age}" in workout_entry.get("message", "")
     and f"Gender: {profile.gender}" in workout_entry.get("message", "")),
    ("R11 Workout Scheduler produced a weekly plan (5+ day names)", sum(d in text(workout_plan) for d in days) >= 5),
    ("R12 Final Markdown plan contains BMI insights, diet and fitness schedule",
     all(part and part in final_plan for part in (bmi_insights, meal_plan, workout_plan)) and final_plan.startswith("# ")),
]

for name, ok in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
print(f"\n{sum(ok for _, ok in checks)}/{len(checks)} checks passed")